In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import os
import sklearn

In [2]:
sklearn.set_config(transform_output="pandas")

In [3]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
X_train = train.drop(['id', 'addicted_label'], axis=1)
y_train = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [5]:
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['number']).columns.tolist()

/tmp/ipykernel_10366/1780040723.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


In [6]:
# Preprocessing Pipeline
imputer = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_cols),
        ('cat', SimpleImputer(strategy='most_frequent'), categorical_cols)
    ],
    verbose_feature_names_out=False
)

In [7]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X_out = X.copy()
        denom_screen = X_out['daily_screen_time_hours'].replace(0, 0.001)
        denom_notif = X_out['notifications_per_day'].replace(0, 0.001)
        X_out['social_media_ratio'] = X_out['social_media_hours'] / denom_screen
        X_out['gaming_ratio'] = X_out['gaming_hours'] / denom_screen
        X_out['work_study_ratio'] = X_out['work_study_hours'] / denom_screen
        X_out['app_opens_per_hour'] = X_out['app_opens_per_day'] / denom_screen
        X_out['notifications_to_opens_ratio'] = X_out['app_opens_per_day'] / denom_notif
        X_out['sleep_deficit'] = 8.0 - X_out['sleep_hours']
        return X_out

In [8]:
final_preprocessor = ColumnTransformer(
    transformers=[('cat_encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [9]:
preprocessor = Pipeline([
    ('imputer', imputer),
    ('engineer', FeatureEngineer()),
    ('final_preprocessor', final_preprocessor)
])

In [10]:
print("Preprocessing data...")
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

Preprocessing data...


In [11]:
for col in categorical_cols:
    X_train_prep[col] = X_train_prep[col].astype('category')
    X_test_prep[col] = X_test_prep[col].astype('category')

In [12]:
print("Starting Adversarial Validation...")
X_train_adv = X_train_prep.copy()
X_test_adv = X_test_prep.copy()

Starting Adversarial Validation...


In [13]:
X_train_adv['is_test'] = 0
X_test_adv['is_test'] = 1
X_adv = pd.concat([X_train_adv, X_test_adv], axis=0, ignore_index=True)
y_adv = X_adv['is_test']
X_adv = X_adv.drop('is_test', axis=1)

In [14]:
adv_model = LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
adv_model.fit(X_adv, y_adv)

,learning_rate,0.05
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [15]:
adv_preds = adv_model.predict_proba(X_adv)[:, 1]
adv_auc = roc_auc_score(y_adv, adv_preds)
print(f"Adversarial Validation AUC: {adv_auc:.4f}")

Adversarial Validation AUC: 0.6760


In [16]:
# Find drifting features
feature_imp = pd.DataFrame({'feature': X_adv.columns, 'importance': adv_model.feature_importances_})
feature_imp = feature_imp.sort_values(by='importance', ascending=False)
print("Top 5 drifting features:")
print(feature_imp.head(5))

Top 5 drifting features:
                    feature  importance
14         work_study_ratio         518
12       social_media_ratio         434
13             gaming_ratio         359
4   daily_screen_time_hours         273
6              gaming_hours         221


In [17]:
features_to_drop = []
if adv_auc > 0.6:  # Only drop if drift is significant
    print("Significant drift detected. Dropping top 2 drifting features...")
    features_to_drop = feature_imp['feature'].head(2).tolist()
    X_train_prep = X_train_prep.drop(features_to_drop, axis=1)
    X_test_prep = X_test_prep.drop(features_to_drop, axis=1)
else:
    print("No significant drift detected. Keeping all features.")

Significant drift detected. Dropping top 2 drifting features...


In [18]:
print("Training final model on non-drifting features...")
model = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
model.fit(X_train_prep, y_train)

Training final model on non-drifting features...


,learning_rate,0.05
,n_estimators,200
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [19]:
print("Predicting final results...")
final_preds = model.predict_proba(X_test_prep)[:, 1]

Predicting final results...


In [20]:
os.makedirs('submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('submissions/adversarial_validation.csv', index=False)
print("Submission saved to submissions/adversarial_validation.csv")

Submission saved to submissions/adversarial_validation.csv
